In [30]:
import seaborn as sns
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

In [32]:
df=sns.load_dataset('tips')

In [34]:
df

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2


In [36]:
X=df.drop(labels='total_bill',axis=1)

In [38]:
X

,tip,sex,smoker,day,time,size
0,1.01,Female,No,Sun,Dinner,2
1,1.66,Male,No,Sun,Dinner,3
2,3.50,Male,No,Sun,Dinner,3
3,3.31,Male,No,Sun,Dinner,2
4,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...
239,5.92,Male,No,Sat,Dinner,3
240,2.00,Female,Yes,Sat,Dinner,2
241,2.00,Male,Yes,Sat,Dinner,2
242,1.75,Male,No,Sat,Dinner,2


In [40]:
y=df['total_bill']

In [42]:
y

0      16.99
1      10.34
2      21.01
3      23.68
4      24.59
       ...  
239    29.03
240    27.18
241    22.67
242    17.82
243    18.78
Name: total_bill, Length: 244, dtype: float64

In [44]:
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42,test_size=.20)

In [46]:
X_train.shape

(195, 6)

In [48]:
X_test.shape

(49, 6)

In [50]:
categorical_cols=['sex','smoker','day','time']
numerical_cols=['tip','size']

In [60]:
#catergorical 

cat_pipline=Pipeline(
    [
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('onehotencoder',OneHotEncoder())
    ]
)

num_pipline=Pipeline(
    [
       ('imputer',SimpleImputer(strategy='median')),
        ('scaler',StandardScaler())
        
    ]
)

In [62]:
preprocessor=ColumnTransformer(
    [
        ('num_pipline',num_pipline,numerical_cols),
        ('cat_pipline',cat_pipline,categorical_cols)
    ]
)

In [64]:
X_train=preprocessor.fit_transform(X_train)
X_test=preprocessor.fit_transform(X_test)

In [78]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score

In [80]:
models={
    'Random forest':RandomForestRegressor(),
    'Linear Regression':LinearRegression(),
    'Decision Tree':DecisionTreeRegressor()
    
}

In [86]:
# model training


def evaluate_model(X_train,X_test,y_train,y_test,models):
    report={}
    for i in range(len(models)):
        model=list(models.values())[i]
        model.fit(X_train,y_train)
        
        y_test_pred=model.predict(X_test)
        test_model=r2_score(y_test,y_test_pred)
        report[list(models.keys())[i]]=test_model      
    return report

In [88]:
evaluate_model(X_train,X_test,y_train,y_test,models)

{'Random forest': 0.5320672296700579,
 'Linear Regression': 0.6057856225746671,
 'Decision Tree': 0.23880330178177922}

In [90]:
## Hyperparameter tunning

In [110]:
para={
    'max_depth':[3,5,10,None],
   'n_estimators':[100,200,300],
    'criterion':['squared_error', 'absolute_error']

}

In [112]:
rand_hp=RandomForestRegressor()

In [114]:
from sklearn.model_selection import GridSearchCV

In [116]:
grd=GridSearchCV(rand_hp,param_grid=para,verbose=3,cv=5,scoring='r2')

In [118]:
grd.fit(X_train,y_train)

Fitting 5 folds for each of 24 candidates, totalling 120 fits
[CV 1/5] END criterion=squared_error, max_depth=3, n_estimators=100;, score=0.293 total time=   0.0s
[CV 2/5] END criterion=squared_error, max_depth=3, n_estimators=100;, score=0.688 total time=   0.0s
[CV 3/5] END criterion=squared_error, max_depth=3, n_estimators=100;, score=0.531 total time=   0.0s
[CV 4/5] END criterion=squared_error, max_depth=3, n_estimators=100;, score=0.413 total time=   0.0s
[CV 5/5] END criterion=squared_error, max_depth=3, n_estimators=100;, score=0.199 total time=   0.0s
[CV 1/5] END criterion=squared_error, max_depth=3, n_estimators=200;, score=0.283 total time=   0.1s
[CV 2/5] END criterion=squared_error, max_depth=3, n_estimators=200;, score=0.701 total time=   0.1s
[CV 3/5] END criterion=squared_error, max_depth=3, n_estimators=200;, score=0.545 total time=   0.1s
[CV 4/5] END criterion=squared_error, max_depth=3, n_estimators=200;, score=0.405 total time=   0.1s
[CV 5/5] END criterion=square

GridSearchCV(cv=5, estimator=RandomForestRegressor(),
             param_grid={'criterion': ['squared_error', 'absolute_error'],
                         'max_depth': [3, 5, 10, None],
                         'n_estimators': [100, 200, 300]},
             scoring='r2', verbose=3)

In [120]:
grd.best_params_

{'criterion': 'squared_error', 'max_depth': 5, 'n_estimators': 100}

In [122]:
y_pred=grd.predict(X_test)

In [124]:
r2_score(y_test,y_pred)

0.576762510330497

In [ ]:
52%  to 57 % 